In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval[onnx,opencv] maite-datasets torchvision
except Exception:
    pass

In [2]:
import warnings
from collections.abc import Callable
from pathlib import Path

import numpy as np
import torch
from maite_datasets.image_classification import MilitaryVehicles
from torchvision.models import ResNet18_Weights, resnet18

from dataeval import Embeddings, Metadata
from dataeval.data import Limit, Resize, Shuffle, View
from dataeval.extractors import BoVWExtractor, FlattenExtractor, OnnxExtractor, TorchExtractor
from dataeval.utils.onnx import to_encoding_model

In [3]:
dataset = MilitaryVehicles("./data", image_set="test", download=True, lazy=True)
subset = View(dataset, [Shuffle(seed=0), Limit(300)])
labels = Metadata(subset).class_labels

print(f"Images: {len(subset)}")
print(f"Shape of the first image: {subset[0][0].shape}")

Images: 300
Shape of the first image: (3, 183, 275)


In [4]:
weights = ResNet18_Weights.DEFAULT
torch_extractor = TorchExtractor(
    resnet18(weights=weights, progress=False),
    transforms=weights.transforms(),
    layer_name="avgpool",
)
torch_embeddings = Embeddings(subset, extractor=torch_extractor, batch_size=32)

print(f"Embeddings shape: {torch_embeddings.shape}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


Embeddings shape: (300, 512)


In [5]:
model_path = Path("data") / "resnet18.onnx"
model_path.parent.mkdir(exist_ok=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    torch.onnx.export(
        resnet18(weights=weights, progress=False).eval(),
        (torch.zeros(1, 3, 224, 224),),
        str(model_path),
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=13,
        dynamo=False,
    )

In [6]:
encoding_model, embedding_layer = to_encoding_model(model_path)

print(f"Embedding layer: {embedding_layer}")

Embedding layer: /Flatten_output_0


In [7]:
def numpy_transform(transform: Callable[[torch.Tensor], torch.Tensor]) -> Callable[[np.ndarray], np.ndarray]:
    """Run a torchvision preprocessing transform on one NumPy image and return a NumPy array."""

    def apply(image: np.ndarray) -> np.ndarray:
        return transform(torch.as_tensor(np.asarray(image))).numpy()

    return apply


onnx_extractor = OnnxExtractor(
    encoding_model,
    transforms=numpy_transform(weights.transforms()),
    output_name=embedding_layer,
)
onnx_embeddings = Embeddings(subset, extractor=onnx_extractor, batch_size=32)

print(f"Embeddings shape: {onnx_embeddings.shape}")

Embeddings shape: (300, 512)


In [8]:
# TEST ASSERTION CELL ###
# the ONNX path reproduces the PyTorch embeddings
torch_array = np.asarray(torch_embeddings)
onnx_array = np.asarray(onnx_embeddings)
cosine = (torch_array * onnx_array).sum(axis=1) / (
    np.linalg.norm(torch_array, axis=1) * np.linalg.norm(onnx_array, axis=1)
)
assert torch_array.shape == onnx_array.shape == (300, 512)
assert cosine.min() > 0.999

In [9]:
resized = View(subset, Resize((128, 128)))

flatten_embeddings = Embeddings(resized, extractor=FlattenExtractor(), batch_size=32)
bovw_extractor = BoVWExtractor(vocab_size=64).fit(resized)
bovw_embeddings = Embeddings(resized, extractor=bovw_extractor, batch_size=32)

print(f"Flatten embeddings shape: {flatten_embeddings.shape}")
print(f"BoVW embeddings shape: {bovw_embeddings.shape}")

Flatten embeddings shape: (300, 49152)
BoVW embeddings shape: (300, 64)


In [10]:
# TEST ASSERTION CELL ###
assert flatten_embeddings.shape == (300, 3 * 128 * 128)
assert bovw_embeddings.shape == (300, 64)

In [11]:
def same_class_neighbors(embeddings: Embeddings) -> float:
    """Share of the images whose nearest other image has the same class."""
    unit = np.asarray(embeddings, dtype=np.float64)
    unit /= np.linalg.norm(unit, axis=1, keepdims=True)
    similarity = unit @ unit.T
    np.fill_diagonal(similarity, -np.inf)
    return float((labels[similarity.argmax(axis=1)] == labels).mean())


chance = float(((np.bincount(labels) / len(labels)) ** 2).sum())
rates = {
    "ResNet18, PyTorch": same_class_neighbors(torch_embeddings),
    "ResNet18, ONNX": same_class_neighbors(onnx_embeddings),
    "Flattened pixels": same_class_neighbors(flatten_embeddings),
    "BoVW": same_class_neighbors(bovw_embeddings),
}

print(f"{'By chance':>17}: {chance:.1%}")
for name, rate in rates.items():
    print(f"{name:>17}: {rate:.1%}")

        By chance: 5.6%
ResNet18, PyTorch: 26.3%
   ResNet18, ONNX: 26.3%
 Flattened pixels: 6.0%
             BoVW: 3.0%


In [12]:
# TEST ASSERTION CELL ###
assert len(set(labels.tolist())) == 24  # the sample covers every class
assert abs(chance - 0.056) < 0.005
assert 4 < rates["ResNet18, PyTorch"] / chance < 6  # "about five times"
assert abs(rates["ResNet18, ONNX"] - rates["ResNet18, PyTorch"]) < 0.01  # "matches it"
assert rates["Flattened pixels"] < 1.5 * chance
assert rates["BoVW"] < 0.5 * rates["ResNet18, PyTorch"]  # BoVW varies between runs, so compare, not pin